# MARS-SOHO Phase 1B — train-only allocation repair

Run CIFAR-100 first. This notebook never extracts or opens test features. It compares exact replay, uniform heterogeneous moment replay, continuous Top-K turnover allocation, and sufficient-statistic-variance allocation with matched shuffled controls. SRQ is disabled.

In [ ]:
# === Edit DATASET_KEY only when starting a separately reviewed run. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'experiment/soho-selfcontained'
DATASET_KEY = 'cifar100'
WORK_DIR = '/content/SOHO-CL'
FEATURE_CACHE_ROOT = '/content/mars_soho_phase1_features'
OUTPUT_ROOT = '/content/mars_soho_phase1b_outputs'
BATCH_SIZE = 128
NUM_WORKERS = 2
EXPECTED_CONFIG_SHA256 = '114662ab2eadcf6003c5e15d0c71c841af688f5a53fbf13bb388fc32b36719e8'
EXPECTED_RUNNER_SHA256 = '1f800e2eeb0b6e09185d2c1c112b3f0b6c1058a0b922f729d50c6f366a08f8cf'

In [ ]:
# Fresh immutable checkout and locked-source verification.
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
os.chdir('/content')
repo = Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub','pandas','matplotlib','seaborn'], check=True)
import torch
assert torch.cuda.is_available(), 'Select Runtime -> Change runtime type -> T4 GPU.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
CONFIG = 'configs/mars_soho_phase1b_train_only.json'
RUNNER = 'tools/mars_soho_phase1b.py'
assert sha(CONFIG) == EXPECTED_CONFIG_SHA256, 'Config hash mismatch'
assert sha(RUNNER) == EXPECTED_RUNNER_SHA256, 'Runner hash mismatch'
assert not subprocess.check_output(['git','status','--porcelain'], text=True).strip()
print('GPU:', torch.cuda.get_device_name(0))
print('commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print('MARS-SOHO PHASE-1B SOURCE CHECK: PASS')

In [ ]:
# Download the verified frozen ViT checkpoint and CIFAR-100 only.
import kagglehub
from huggingface_hub import hf_hub_download
assert DATASET_KEY == 'cifar100', 'Phase 1B must be reviewed on CIFAR-100 first.'
CHECKPOINT_PATH = hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size == 346284714
assert sha(CHECKPOINT_PATH) == '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
DATASET_ROOT = kagglehub.dataset_download('zaphat206/cifar-100')
print('dataset:', DATASET_KEY, DATASET_ROOT)

In [ ]:
# Restore the runtime train cache if present; otherwise extract TRAIN features with live progress.
protocol = json.loads(Path(CONFIG).read_text())
dataset = protocol['datasets'][DATASET_KEY]
cache = Path(FEATURE_CACHE_ROOT) / DATASET_KEY
if not (cache/'train.pt').is_file():
    command = [sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',DATASET_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256',protocol['backbone']['checkpoint_sha256'],'--feature-cache-dir',str(cache),'--output-dir',f'/content/unused_mars_phase1b_{DATASET_KEY}','--dataset',dataset['dataset'],'--model-name',protocol['backbone']['model_name'],'--data-augmentation','vit','--seed','2025','--num-classes',str(dataset['num_classes']),'--num-tasks',str(dataset['num_tasks']),'--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('TRAIN FEATURE EXTRACTION START. One progress line is printed per task.', flush=True)
    subprocess.run(command, check=True)
else:
    print('Using existing runtime TRAIN cache:', cache)
assert (cache/'train.pt').is_file() and (cache/'metadata.json').is_file()
assert not (cache/'test.pt').exists(), 'FAIL: test.pt became visible'
print('TRAIN CACHE READY | test.pt absent')

In [ ]:
# Mathematical, state, anti-collapse and tiny-runner correctness gate.
tests = ['tests/test_mars_soho_math.py','tests/test_mars_soho_learner.py','tests/test_mars_soho_phase1.py','tests/test_mars_soho_phase1b.py']
completed = subprocess.run([sys.executable,'-B','-m','pytest','-q',*tests], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(completed.stdout, flush=True)
assert completed.returncode == 0, 'Correctness gate failed.'
print('MARS-SOHO PHASE-1B CORRECTNESS GATE: PASS')

## Locked train-only Phase 1B
The runner executes 6 methods × 3 paired replicates. `START` begins a unit, every `TASK` line completes a continual stage, and `DONE` makes that unit resumable on the current runtime disk. The run uses fresh train-only validation splits and performs no hyperparameter search.

In [ ]:
# Start/resume Phase 1B. No held-out test feature or label is opened.
command = [sys.executable,'-u',RUNNER,'--config',CONFIG,'--dataset-key',DATASET_KEY,'--feature-cache-dir',str(cache),'--output-root',OUTPUT_ROOT,'--device','cuda']
print('STARTING PHASE 1B: 18 paired method/replicate units.', flush=True)
started = time.time()
completed = subprocess.run(command)
print(f'elapsed={(time.time()-started)/60:.1f} minutes | return_code={completed.returncode}', flush=True)
assert completed.returncode == 0, 'Runner failed; return the full traceback without editing config.'
RESULT_PATH = Path(OUTPUT_ROOT)/DATASET_KEY/'phase1b_results.json'
assert RESULT_PATH.is_file()
print('PHASE-1B PROCESS COMPLETE')

In [ ]:
# Inspect accuracy and verify that risk/allocation no longer collapse.
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
payload = json.loads(RESULT_PATH.read_text())
rows = [{'method':method,'outer_validation_AIA':score} for method,score in payload['outer_mean_aia'].items()]
table = pd.DataFrame(rows).sort_values('outer_validation_AIA', ascending=False)
display(table)
print('allocation diagnostics:', json.dumps(payload['allocation_diagnostics'], indent=2))
print('gates:', json.dumps(payload['gates'], indent=2))
ax=sns.barplot(data=table,x='method',y='outer_validation_AIA'); ax.tick_params(axis='x',rotation=35); ax.set_title('CIFAR-100 fresh train-only validation AIA'); plt.tight_layout(); plt.show()
risk_rows=[]
for ridx,result in enumerate(payload['outer_validation']['statistic_variance_aware']):
    for task,diag in enumerate(result['task_diagnostics'],1):
        risks=diag.get('pilot_risks',{}).get('statistic_variance',{})
        allocations=diag.get('pseudo_allocation',{})
        for class_id,risk in risks.items(): risk_rows.append({'replicate':ridx,'task':task,'class_id':int(class_id),'risk':risk,'allocation':allocations.get(class_id,allocations.get(str(class_id)))})
if risk_rows:
    risk_df=pd.DataFrame(risk_rows); display(risk_df[['risk','allocation']].describe())
    sns.scatterplot(data=risk_df,x='risk',y='allocation',hue='replicate',alpha=.5); plt.title('Statistic variance vs fixed-budget allocation'); plt.show()

In [ ]:
# Export evidence only; per-sample frozen-feature cache is excluded.
from google.colab import files
evidence = Path(OUTPUT_ROOT)/DATASET_KEY
shutil.copy2(CONFIG, evidence/'locked_config.json')
shutil.copy2(RUNNER, evidence/'locked_runner.py')
for relative in ['methods/mars_soho/geometry.py','methods/mars_soho/reconstruction.py','methods/mars_soho/learner.py']:
    shutil.copy2(relative, evidence/Path(relative).name)
archive=shutil.make_archive('/content/mars_soho_phase1b_cifar100_train_only','zip',root_dir=evidence)
print('artifact:',archive,'bytes=',Path(archive).stat().st_size,'sha256=',sha(archive))
files.download(archive)